In [24]:
import pandas as pd

# Datei laden
df = pd.read_csv("../../documentation/clean_entity_mapping.csv")

# Nur relevante Spalten auswählen: pdb, heavy_subclass und H_entity_id
df = df[["pdb", "heavy_subclass", "H_entity_id"]].dropna(subset=["pdb", "heavy_subclass", "H_entity_id"])

# Einheitliche Formatierung
df["pdb"] = df["pdb"].str.lower()
df["heavy_subclass"] = df["heavy_subclass"].str.upper().str.strip()

# H_entity_id sicherstellen, dass sie int ist
df["H_entity_id"] = df["H_entity_id"].astype(int)

# Genfamilien numerisch als "Cluster" codieren
df["genfamily_cluster"] = pd.factorize(df["heavy_subclass"])[0] + 1  # Cluster-Index ab 1

# Speichern für Vergleich mit Sequenz-Clustering
df[["pdb", "H_entity_id", "genfamily_cluster"]].to_csv("genfamily_clusters.csv", index=False)

# Optional anzeigen
print(df["heavy_subclass"].value_counts())
print(f"{df['genfamily_cluster'].nunique()} Genfamilien-Cluster gefunden.")

heavy_subclass
IGHV3     1307
IGHV1      774
IGHV4      219
IGHV5      157
IGHV2       83
IGHV7       18
IGHV6       15
IGHV14      14
IGHV9        2
Name: count, dtype: int64
9 Genfamilien-Cluster gefunden.


nun habe ich die Genfamilen geclustert.

Vergleich der custer der aa Sequenzen und den Genfamilien.

In [35]:
import pandas as pd
from sklearn.metrics import adjusted_rand_score

# Hierarchische Cluster-Datei laden
hier_clusters = pd.read_csv("pca_with_clusters.csv", index_col=0)

# Index (z.B. '1ADQ_entity3_chainH') aufsplitten in pdb, entity, chain
hier_clusters = hier_clusters.reset_index()
hier_clusters[['pdb', 'entity', 'chain']] = hier_clusters['index'].str.split('_', expand=True)



# Entity-ID extrahieren (z.B. 'entity3' → 3)
hier_clusters['entity_id'] = hier_clusters['entity'].str.extract('(\d+)').astype(int)

# Genfamilien-Datei laden
gen_clusters = pd.read_csv("genfamily_clusters.csv")

# Vor dem Casten: NAs entfernen
gen_clusters = gen_clusters[gen_clusters['H_entity_id'].notna()]
gen_clusters['H_entity_id'] = gen_clusters['H_entity_id'].astype(int)

hier_clusters['pdb'] = hier_clusters['pdb'].str.lower()
gen_clusters['pdb'] = gen_clusters['pdb'].str.lower()

# Merge über pdb und entity_id
merged = pd.merge(
    hier_clusters,
    gen_clusters,
    left_on=['pdb', 'entity_id'],
    right_on=['pdb', 'H_entity_id']
)

print(f"Gematchte Einträge: {len(merged)}")

# Cluster-Labels und Genfamilien als numerische Kategorien
labels_seq = merged['cluster']  # z.B. aus pca_with_clusters.csv
labels_gen = merged['genfamily_cluster']  # aus genfamily_clusters.csv, numerisch codiert


# ARI berechnen
ari = adjusted_rand_score(labels_gen, labels_seq)
print(f"Adjusted Rand Index (ARI): {ari:.3f}")

# NMI berechnen
nmi = normalized_mutual_info_score(labels_gen, labels_seq)
print(f"Normalized Mutual Information (NMI): {nmi:.3f}")


Gematchte Einträge: 2589
Adjusted Rand Index (ARI): 0.002
Normalized Mutual Information (NMI): 0.307


In [34]:
import pandas as pd
from sklearn.metrics import adjusted_rand_score

# Hierarchische Cluster-Datei laden
hier_clusters = pd.read_csv("pca_kmeans_clusters.csv", index_col=0)

# Index (z.B. '1ADQ_entity3_chainH') aufsplitten in pdb, entity, chain
hier_clusters = hier_clusters.reset_index()
hier_clusters[['pdb', 'entity', 'chain']] = hier_clusters['index'].str.split('_', expand=True)



# Entity-ID extrahieren (z.B. 'entity3' → 3)
hier_clusters['entity_id'] = hier_clusters['entity'].str.extract('(\d+)').astype(int)

# Genfamilien-Datei laden
gen_clusters = pd.read_csv("genfamily_clusters.csv")

# Vor dem Casten: NAs entfernen
gen_clusters = gen_clusters[gen_clusters['H_entity_id'].notna()]
gen_clusters['H_entity_id'] = gen_clusters['H_entity_id'].astype(int)

hier_clusters['pdb'] = hier_clusters['pdb'].str.lower()
gen_clusters['pdb'] = gen_clusters['pdb'].str.lower()

# Merge über pdb und entity_id
merged = pd.merge(
    hier_clusters,
    gen_clusters,
    left_on=['pdb', 'entity_id'],
    right_on=['pdb', 'H_entity_id']
)

print(f"Gematchte Einträge: {len(merged)}")

# Cluster-Labels und Genfamilien als numerische Kategorien
labels_seq = merged['cluster']  # z.B. aus pca_with_clusters.csv
labels_gen = merged['genfamily_cluster']  # aus genfamily_clusters.csv, numerisch codiert


# ARI berechnen
ari = adjusted_rand_score(labels_gen, labels_seq)
print(f"Adjusted Rand Index (ARI): {ari:.3f}")

# NMI berechnen
nmi = normalized_mutual_info_score(labels_gen, labels_seq)
print(f"Normalized Mutual Information (NMI): {nmi:.3f}")

Gematchte Einträge: 2589
Adjusted Rand Index (ARI): 0.150
Normalized Mutual Information (NMI): 0.235
